# Forecasting and Time Series for Decisions

Notebook for **Chapter 9** of the *ds* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/ds/ch09.html

Tip: change a number, predict the output, then run the cell again.

## 9.1 Components, Baselines and Error Metrics

## 9.2 Stationarity, ACF and the Classical Models

## 9.3 Machine Learning with Lags, and How to Validate

In [ ]:
# one-step features: lag1, lag7, lag14, lag28, r7, r28, dow, doy
for tri, tei in KFold(5, shuffle=True, random_state=0).split(X): ...   # random folds
m = HistGradientBoostingRegressor(...).fit(X.iloc[:600], np.log(y.iloc[:600]))  # time split

## 9.4 Intervals, Special Cases and Forecast to Decision

## 9.5 Hierarchies: Making Forecasts Add Up

A retailer forecasts sales for each store, and also for the whole company. If you build each forecast on its own, the store forecasts will not add up to the company forecast. Finance sees two different totals and loses trust. **Forecast reconciliation** adjusts all the forecasts together so that parts sum to the whole, and, done well, it also improves accuracy.

In [ ]:
import numpy as np
# Part 1: tiny hand example. Total = A + B, but base forecasts disagree.
S = np.array([[1, 1], [1, 0], [0, 1]], float)           # rows: Total, A, B
yhat = np.array([105., 40., 55.])
bu = S @ yhat[1:]
G = np.linalg.inv(S.T @ S) @ S.T                          # OLS reconciliation
ols = S @ G @ yhat
print("base       ", yhat, " sum of parts =", yhat[1] + yhat[2])
print("bottom-up  ", bu)
print("OLS recon  ", np.round(ols, 3))
# Part 2: simulation. 6 stores, shared weekly pattern; base forecasts are noisy and incoherent.
rng = np.random.default_rng(2)
m, T, H = 6, 140, 28
level = rng.uniform(40, 120, m)
t = np.arange(T + H)
dow = np.array([0.9, 0.95, 1.0, 1.0, 1.05, 1.2, 0.9])[t % 7]
Y = level[:, None] * dow * (1 + 0.002*t) + rng.normal(0, 1, (m, T + H)) * np.sqrt(level)[:, None] * 1.5
tot = Y.sum(0)
def seas_naive(y):                                        # last observed week repeated
    return np.tile(y[T - 7:T], H // 7)
def ma_scaled(y):                                         # smooth level times weekday shape
    return y[T - 28:T].mean() * dow[T:T + H] / dow[T - 28:T].mean()
base_b = np.array([seas_naive(Y[i]) for i in range(m)])
base_t = ma_scaled(tot)                                   # a different method for the total, so totals disagree
Sfull = np.vstack([np.ones(m), np.eye(m)])
yh = np.vstack([base_t, base_b])                          # (1+m) x H
G = np.linalg.inv(Sfull.T @ Sfull) @ Sfull.T
rec = Sfull @ G @ yh
bu = Sfull @ base_b
actual = np.vstack([tot[T:], Y[:, T:]])
def mae(f, rows): return np.abs(f[rows] - actual[rows]).mean()
print(f"{'method':12s}{'total MAE':>10s}{'store MAE':>10s}")
for name, f in [("base", yh), ("bottom-up", bu), ("OLS recon", rec)]:
    print(f"{name:12s}{mae(f, [0]):10.2f}{mae(f, list(range(1, m + 1))):10.2f}")
print("gap in base total vs sum of stores (mean abs):", round(np.abs(yh[0] - yh[1:].sum(0)).mean(), 2))
print("gap after reconciliation:", round(np.abs(rec[0] - rec[1:].sum(0)).max(), 6))

base        [105.  40.  55.]  sum of parts = 95.0
bottom-up   [95. 40. 55.]
OLS recon   [101.667  43.333  58.333]
method       total MAE store MAE
base             31.58     13.56
bottom-up        43.11     13.56
OLS recon        32.68     12.92
gap in base total vs sum of stores (mean abs): 16.34
gap after reconciliation: 0.0


## 9.6 Intermittent Demand: Croston, TSB and the Trap of MAE

Spare parts, slow SKUs and rare events have many days of zero demand and occasional spikes. Standard methods smooth these badly: they forecast a small number every day, and the usual error metric then rewards the wrong answer.

In [ ]:
import numpy as np
rng = np.random.default_rng(4)
n, split = 400, 300
occur = rng.random(n) < 0.2                               # demand happens on 20% of days
size = rng.poisson(4, n) + 1                              # size when it happens, 1 or more
d = np.where(occur, size, 0).astype(float)
train, test = d[:split], d[split:]
print("share of zero days:", round((d == 0).mean(), 3), " mean demand/day:", round(d.mean(), 3))
def croston(y, a=0.1, sba=False):
    z = None; p = None; q = 1
    for v in y:
        if v > 0:
            if z is None: z, p = v, q
            else: z += a*(v - z); p += a*(q - p)
            q = 1
        else: q += 1
    f = z / p
    return f * (1 - a/2) if sba else f
def tsb(y, a=0.1, b=0.1):
    prob = (y > 0).mean(); z = y[y > 0][0]
    for v in y:
        prob += b*((v > 0) - prob)
        if v > 0: z += a*(v - z)
    return prob * z
fc = {"zero (median)": 0.0, "last value": train[-1], "mean of all": train.mean(),
      "moving avg 28": train[-28:].mean(), "Croston": croston(train),
      "Croston SBA": croston(train, sba=True), "TSB": tsb(train)}
print(f"{'method':16s}{'forecast':>9s}{'MAE':>7s}{'RMSE':>7s}{'bias':>8s}")
for k, f in fc.items():
    e = f - test
    print(f"{k:16s}{f:9.3f}{np.abs(e).mean():7.3f}{np.sqrt((e**2).mean()):7.3f}{e.mean():8.3f}")
print("actual mean in test:", round(test.mean(), 3))

share of zero days: 0.838  mean demand/day: 0.848
method           forecast    MAE   RMSE    bias
zero (median)       0.000  1.170  2.696  -1.170
last value          6.000  5.090  5.406   4.830
mean of all         0.740  1.584  2.467  -0.430
moving avg 28       0.571  1.490  2.502  -0.599
Croston             0.726  1.576  2.470  -0.444
Croston SBA         0.689  1.556  2.476  -0.481
TSB                 0.796  1.616  2.458  -0.374
actual mean in test: 1.17


## 9.7 Honest Intervals: Conformal Prediction and Pinball Loss

Section 9.4 built intervals from backtest percentiles. **Split conformal prediction** formalises that idea and gives a coverage promise under minimal assumptions. Hold out a calibration set. Compute the absolute errors of the model on it. Take the (1 − α)(n + 1)/n quantile of these errors, call it `q`. The interval is `forecast ± q`. If calibration and future errors are exchangeable, the interval covers the truth at least 90% of the time for α = 0.1.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
rng = np.random.default_rng(6)
n = 500
t = np.arange(n)
y = 200 + 0.2*t + 25*np.sin(2*np.pi*t/7) + rng.normal(0, 8, n) * (1 + 0.004*t)   # noise grows a little
X = np.c_[t, np.sin(2*np.pi*t/7), np.cos(2*np.pi*t/7)]
tr, ca, te = slice(0, 300), slice(300, 400), slice(400, 500)           # train, calibration, test (time order)
m = LinearRegression().fit(X[tr], y[tr])
res = np.abs(y[ca] - m.predict(X[ca]))
k = int(np.ceil((len(res) + 1) * 0.9))
q = np.sort(res)[k - 1]                                  # 90% conformal half-width
pred = m.predict(X[te])
inside = (y[te] >= pred - q) & (y[te] <= pred + q)
print("conformal half-width:", round(q, 2))
print("coverage on test: ", round(inside.mean(), 3), "(target 0.90)")
# naive normal interval from training residual sd
sd = (y[tr] - m.predict(X[tr])).std()
inside_n = np.abs(y[te] - pred) <= 1.645*sd
print("normal interval half-width:", round(1.645*sd, 2), " coverage:", round(inside_n.mean(), 3))
# pinball loss for the 0.9 quantile
def pinball(a, f, qq): e = a - f; return np.mean(np.maximum(qq*e, (qq-1)*e))
up = pred + q
print("pinball q=0.9 for the upper bound:", round(pinball(y[te], up, 0.9), 3))
print("pinball q=0.9 for the point forecast:", round(pinball(y[te], pred, 0.9), 3))
# hand check with 3 numbers
a = np.array([10., 20., 30.]); f = np.array([12., 12., 12.])
print("hand check pinball q=0.9:", round(pinball(a, f, 0.9), 3))

conformal half-width: 31.96
coverage on test:  0.84 (target 0.90)
normal interval half-width: 21.37  coverage: 0.65
pinball q=0.9 for the upper bound: 4.169
pinball q=0.9 for the point forecast: 9.616
hand check pinball q=0.9: 7.867


## Chapter recap

## Practice set

**1. (easy)** Seasonal naive on daily data with m = 7: forecast for next Tuesday if last Tuesday was 120? **Answer:** 120.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/ds/ch09.html